## Import Libraries

In [ ]:
# Import necessary libraries for data handling, optimization, and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random, time, copy
import pickle
from itertools import permutations
from gurobipy import *
from gurobipy import GRB
import os

In [ ]:
# Set your working directory
my_folder = "C:/Users/hyunwoolee/OneDrive - Virginia Tech/Hyunwoo Research/upload"

In [ ]:
SEED = 11
random.seed(SEED)
np.random.seed(SEED)

## General Functions

In [ ]:
def kpg_player_utility(profile, player):
    """
    profile: dict[player -> list of selected items]
    player: int

    Returns u_player(profile) using list membership only.
    """
    my_items = profile[player]

    # direct profit
    util = sum(profits[player][i] for i in my_items)

    # interaction terms where action[0] == player
    # term interactions[(player,j)][i] is earned if i in my_items and i in profile[j]
    for (a0, a1) in players_interaction:
        if a0 != player:
            continue
        other_items = profile[a1]
        # list membership only
        for i in my_items:
            if i in other_items:
                util += interactions[(a0, a1)][i]
    return float(util)


In [ ]:
def from_x_to_obj(profile):
    """
    Computes both individual (selfish) and global objective values.

    Parameters:
        selected_items (dict): Dictionary mapping each player to their selected items.

    Returns:
        selfish_obj (dict): Objective value for each individual player.
        global_obj (float): Total objective value across all players.
    """
    selfish_obj = {p: 0.0 for p in players}

    # direct part
    for p in players:
        selfish_obj[p] += sum(profits[p][i] for i in profile[p])

    # interactions: sum over (a0,a1) and items that are in both players' lists
    for (a0, a1) in players_interaction:
        items0 = profile[a0]
        items1 = profile[a1]
        for i in items0:
            if i in items1:
                selfish_obj[a0] += interactions[(a0, a1)][i]

    global_obj = sum(selfish_obj[p] for p in players)
    return selfish_obj, float(global_obj)


In [ ]:
def printSolution(model,x, selected_items):    
    
    # Check if the model found a feasible solution
    if model.SolCount > 0:
        print('\nObjvalue: %g' % model.ObjVal)
        for player in players:
            for i in items:
                if x[player][i].X > 0.1:
                    #print('item %d for player %d: %g' % (i, player, x[player][i].X))
                    selected_items[player].append(i)
    else:
        pass # No feasible solution found

## Social Welfare Model

In [ ]:
### This function builds and solves a Gurobi model that maximizes social welfare

def SW_model(time_limit, usage):

    start_time = time.time()
    
    # Gurobi Model
    model = Model("SW model")
    
    # Decision variables
    x={}
    for player in players:
        x[player] = {}
        for i in items:
            x[player][i] = model.addVar(vtype = GRB.BINARY,name="x_[%s][%s]"%(str(player),str(i)))
 
    y={}
    for action in players_interaction:
        y[action] = {}
        for i in items:
            y[action][i] = model.addVar(vtype = GRB.BINARY,name="y_[%s][%s]"%(str(action),str(i)))
            
    # Player-level knapsack constraints
    for player in players:
        model.addConstr( quicksum(weights[player][i]*x[player][i] for i in items)  <= budgets[player])
        
    # Logical constraints for interaction variables y[action][i]
    for action in players_interaction:
        for i in items:
            model.addConstr(y[action][i] <= x[action[0]][i])
            model.addConstr(y[action][i] <= x[action[1]][i])
            model.addConstr(y[action][i] >= x[action[0]][i]+x[action[1]][i]-1)
    
    # === Objective function: maximize total social welfare ===
    model.setObjective(quicksum(profits[player][i]*x[player][i] for i in items for player in players) 
                       + quicksum(interactions[action][i]*y[action][i] for i in items for action in players_interaction)
                       ,GRB.MAXIMIZE)

    # Set time limit based on usage type
    if usage == 'OSW':
        model.setParam('TimeLimit',time_limit)
    elif usage == 'OSW_warm':
        model.setParam('TimeLimit',time_limit/6)

    # Solver parameters    
    model.Params.LogToConsole = 0
    model.Params.Threads =  user_Threads

    # Solve the model
    model.optimize()
    runtime=model.Runtime
    
    # Extract selected items from solution
    selected_items = {player: [] for player in players}
    printSolution(model,x, selected_items)
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    return model, elapsed_time, selected_items

## Best-response problem

In [ ]:
def selfish_KPG(player_fixed):
    """
    MILP best-response for player_fixed:
    max  sum_i profits[player_fixed][i] x_i + sum_{j!=i} sum_{k in S_j} interactions[(i,j)][k] x_k
    s.t. sum_i weights[player_fixed][i] x_i <= budgets[player_fixed]
         x_i in {0,1}
    """
    m = Model(f"KPG_BR_{player_fixed}")
    m.Params.LogToConsole = 0
    m.Params.OutputFlag = 0
    m.Params.Threads = user_Threads
    m.Params.MIPGap = 1e-6

    x = {i: m.addVar(vtype=GRB.BINARY, name=f"x[{i}]") for i in items}
    m.addConstr(quicksum(weights[player_fixed][i] * x[i] for i in items) <= budgets[player_fixed],
                name="Budget")

    # store handles
    m._x = x
    m._player = player_fixed

    # placeholder objective (we will overwrite each solve)
    m.setObjective(0.0, GRB.MAXIMIZE)
    m.update()
    return m


def update_kpg_br_objective(m, profile):
    """
    Update BR objective of model m (for fixed player i) given current profile of others.
    Uses list membership only.
    """
    i = m._player
    x = m._x

    obj = quicksum(profits[i][k] * x[k] for k in items)

    # add interaction coefficients based on others' selected items
    for (a0, a1) in players_interaction:
        if a0 != i:
            continue
        # if item k is selected by player a1, then interaction contributes interactions[(i,a1)][k] * x[k]
        other_items = profile[a1]
        for k in other_items:
            obj += interactions[(a0, a1)][k] * x[k]

    m.setObjective(obj, GRB.MAXIMIZE)
    m.update()


## RRR-BRD algorithm

In [ ]:
def RRR_BRD(x_current, selfish_model, max_iteration):
    """
    Executes the Random-Restart Random-order Best-Response Dynamics (RRR-BRD) algorithm for the KPG game.

    Starting from an initial strategy profile, this function iteratively updates each player's strategy
    using best-response optimization. Within each initialization, every BRD round uses a fresh random
    permutation of the players. Across initializations, the algorithm restarts from independently generated
    feasible profiles. The procedure searches for a Pure Nash Equilibrium (PNE) but does not perform any
    explicit cycle detection; it simply stops when a fixed point (no player changes strategy in a full round)
    is reached or when the iteration / restart limits are exhausted.

    Args:
        x_current (dict): Initial strategy profile; mapping from player to list of selected items.
        selfish_model (dict or None): Dictionary of pre-built Gurobi models for each player's
            best-response problem. If None, models will be built on demand.
        max_init (int): Maximum number of randomized initial profiles (restarts) to explore.
        max_iteration (int): Maximum number of BRD rounds per initialization.

    Returns:
        tuple:
            selected_items (dict[int, dict]): History of strategy profiles indexed by BRD round;
                for each round `t`, `selected_items[t][player]` is the list of items chosen by `player`.
            BR_PNE_found (bool): True if a Pure Nash Equilibrium was found in any initialization.
            total_iterations (int): Total number of BRD rounds performed over all initializations.
            solution (dict): Utility values for each player under the final profile of the last round.
    """

    global BR_PNE_found

    # Initialize selected_items and solution dictionary
    selected_items = {}
    selected_items[0] = {}
    for player in players:
        selected_items[0][player] = [i for i in x_current[player]]        
    solution = {}
    solution[0] = from_x_to_obj(x_current)[0]

    # Create a container for selfish models if not already provided
    if selfish_model == None:
        selfish_model = {}

    # Initial settings
    TOL = 1e-8
    total_iterations = 0
    
    # Enable random play order    
    players_tmp = players.copy()
    random.shuffle(players_tmp)
    playing_sequence = players_tmp
        
    # Run best-response updates up to max_iteration times
    for num in range(1,max_iteration+1):
        total_iterations += 1

        selected_items[num] = {}
        solution[num] = {}
        for player in players:
            solution[num][player] = 0
            selected_items[num][player] = selected_items[num-1][player]

        for player in playing_sequence:
            # Reuse or initialize best-response model for this player
            if player not in selfish_model:
                selfish_model[player] = selfish_KPG(player)
            m = selfish_model[player]

            # incumbent utility under current profile
            u_inc = float(kpg_player_utility(selected_items[num], player))

            # update BR objective using others' current selections
            update_kpg_br_objective(m, selected_items[num])

            # warm start (list membership only)
            incumbent_list = selected_items[num][player]
            for k in items:
                m._x[k].start = 1 if k in incumbent_list else 0

            m.optimize()
            u_br = float(m.ObjVal)

            if u_br > u_inc + TOL:
                selected_items[num][player] = [k for k in items if m._x[k].X > 0.5]
                solution[num][player] = u_br
            else:
                selected_items[num][player] = incumbent_list
                solution[num][player] = u_inc
                                
        # Check for convergence to a Pure Nash Equilibrium
        if selected_items[len(selected_items)-1] == selected_items[len(selected_items)-2]:
            # print(' PNE found at %d iteration ================================================================== '%(num))
            BR_PNE_found = True
            selected_items_final = selected_items[num]

            return selected_items, BR_PNE_found, total_iterations, solution[num]  
    
        # Enable random play order
        players_tmp = players.copy()
        random.shuffle(players_tmp)
        playing_sequence = players_tmp

    # No equilibrium found after all initializations and iterations        
    return selected_items, False, total_iterations, {player:0 for player in players}     

In [ ]:
def kpg_br_value(player_fixed, profile_dict, selfish_model):
    """Solve player_fixed BR against profile_dict (others fixed) using the player-only MILP BR model."""
    if player_fixed not in selfish_model:
        selfish_model[player_fixed] = selfish_KPG(player_fixed)  # player-only MILP
    m = selfish_model[player_fixed]

    # update BR objective using others' current selections
    selected_self = set(profile_dict[player_fixed])
    update_kpg_br_objective(m, profile_dict)

    # warm start (optional)
    for item in items:
        m._x[item].start = 1 if item in selected_self else 0

    m.update()
    m.optimize()
    return float(m.ObjVal)



def kpg_alpha_end_of_rounds(selected_items_hist, max_round=20, selfish_model=None):
    
    eps = 1e-5
    
    if selfish_model is None:
        selfish_model = {}

    R = min(max_round, max(selected_items_hist.keys()))
    alpha_by_round = {}
    alpha_i_by_round = {}

    best_alpha = float("inf")
    best_round = None
    best_profile = None
    best_global_obj = None

    for t in range(1, R + 1):
        profile = selected_items_hist[t]  # end-of-round profile

        # current utilities + global objective under this profile
        u_curr, global_obj = from_x_to_obj(profile)

        ratios = {}
        for player in players:
            br = kpg_br_value(player, profile, selfish_model)

            ui = float(u_curr[player])
            if ui <= eps:
                # If current utility is ~0:
                # - if best response also ~0 => ratio=1 (no gain)
                # - if best response >0 => ratio=inf (huge multiplicative improvement)
                ratios[player] = 1.0 if br <= eps else float("inf")
            else:
                ratios[player] = br / ui
                
        alpha_t = max(ratios.values())
        alpha_by_round[t] = alpha_t
        alpha_i_by_round[t] = ratios

        if alpha_t < best_alpha:
            best_alpha = alpha_t
            best_round = t
            best_profile = profile
            best_global_obj = float(global_obj)

    return alpha_by_round, alpha_i_by_round, best_alpha, best_round, best_profile, best_global_obj


## Experiment Loop: Run All Settings for KPG Dataset

In [ ]:
standard_time_limit = 1800
user_Threads = 16

non_PNE_instances = [('C', 'cij-sn', 20, 100, 5), ('C', 'cij-sn', 25, 100, 5),('C', 'cij-sn', 25, 100, 8)]


for inst in non_PNE_instances:
    (category, cat_name, inst_players, inst_items, inst_budgets) = inst    
        
    directory_path = f"{my_folder}/BZR_KPG/KPG_generated/type_{category}"
    filenames = os.listdir(directory_path)

    for filename in filenames:
        if filename == f"{inst_players}-{inst_items}-{inst_budgets}-{cat_name}.txt":
        
            print("=" * 100)
            print(f"Processing file: {filename}")
        
            # === Load Instance File ===
            with open(os.path.join(directory_path, filename), 'r') as file:
                lines = file.readlines()
        
            chars = filename.split('-')  # Used for filename parsing
        
            # === Parse Game Parameters ===
            num_players, num_items = map(int, lines[0].split())
            items = list(range(num_items))
            players = list(range(num_players))
            players_complement = {player: [p for p in players if p != player] for player in players}
            budgets = list(map(float, lines[1].split()))
        
            # === Initialize Profits, Weights, and Interactions ===
            profits = {player: {} for player in players}
            weights = {player: {} for player in players}
            players_interaction = list(permutations(players, 2))
            interactions = {action: {} for action in players_interaction}
        
            for line in lines[2:]:
                data = list(map(int, line.split()))
                item = data[0]
        
                for player in players:
                    profits[player][item] = data[2 * player + 1]
                    weights[player][item] = data[2 * player + 2]
        
                offset = 2 * len(players) + 1
                for idx, action in enumerate(players_interaction):
                    interactions[action][item] = data[offset + idx]
                    
    
    
            # === Run RRR_BRD (initial strategy profile: 0) === #
            print("=" * 35 + "  RRR_BRD(0)_model  " + "=" * 35)
            x_current = {player: [] for player in players}
            start_time = time.time()
            
            hist, found, _, sol_last = RRR_BRD(x_current, selfish_model=None, max_iteration=20)

            if found:
                # final profile is hist[max(hist.keys())]
                final_round = max(hist.keys())
                _, best_global = from_x_to_obj(hist[final_round])
                best_alpha = 1.0
                best_round = final_round
            else:
                alpha_round, alpha_i_round, best_alpha, best_round, best_prof, best_global = kpg_alpha_end_of_rounds(hist, max_round=20)
                        
            end_time = time.time()
            time_approx_BRD = end_time - start_time
            

            ##################################  Summarize Results ####################################

            # Create a DataFrame with specified indexes and columns
            indexes1 = ['Total']

            # Create a DataFrame with specified indexes and columns
            columns = ['filename','players','items','budgets','category',
                       'approx_BRD_obj','best_alpha', 'approx_T','best_round']    

            df_tmp_test = pd.DataFrame(index=indexes1, columns=columns)

            df_tmp_test.loc['Total','filename'] = filename
            df_tmp_test.loc['Total','players'] = chars[0]
            df_tmp_test.loc['Total','items'] = chars[1]
            df_tmp_test.loc['Total','budgets'] = chars[2]
            df_tmp_test.loc['Total','category'] = category
            df_tmp_test.loc['Total','approx_BRD_obj'] = best_global
            df_tmp_test.loc['Total','best_alpha'] = best_alpha
            df_tmp_test.loc['Total','approx_T'] = time_approx_BRD
            df_tmp_test.loc['Total','best_round'] = best_round

            # === Save Results to CSV === #
            results_path = f"{my_folder}/BZR_KPG/KPG_results/KPG_approx_test.csv"
            try:
                df_test = pd.read_csv(results_path, index_col=0)
                df_test = pd.concat([df_test, df_tmp_test])
            except FileNotFoundError:
                df_test = df_tmp_test

            df_test.to_csv(results_path)
